# Homework 4 — Training Notebook

**Setup:** In Colab go to `Runtime → Change runtime type → T4 GPU`, then run every cell top-to-bottom.

No Google Drive needed. Code is uploaded directly to the Colab runtime; the dataset is downloaded from the course URL.

> **If you see import errors after Step 2 (pip install):** go to `Runtime → Restart session`, then skip straight to Step 3 and run from there.

## Step 0 — Verify GPU

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [1]:
## ─── QUICK RECONNECT — run this first after any disconnect ───────────────────
import os, sys
from pathlib import Path

PROJECT = '/content/homework4'
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Status check — shows what's already done so you know where to resume
checks = {
    'Code uploaded':       Path('homework/finetune.py').exists(),
    'Dataset downloaded':  Path('data/train').exists() and any(Path('data/train').glob('*_info.json')),
    'QA pairs generated':  Path('data/train/generated_qa_pairs.json').exists(),
    'Captions generated':  Path('data/train/generated_captions.json').exists(),
    'VLM checkpoint':      Path('homework/vlm_model/adapter_config.json').exists(),
    'CLIP checkpoint':     Path('homework/clip_model/adapter_config.json').exists(),
}
for label, ok in checks.items():
    print(f"{'✓' if ok else '✗'}  {label}")
print('\nWorking directory:', os.getcwd())

✓  Code uploaded
✓  Dataset downloaded
✗  QA pairs generated
✗  Captions generated
✗  VLM checkpoint
✗  CLIP checkpoint

Working directory: /content/homework4


In [ ]:
# # Delete stale training data so generation reruns
# import os
# os.remove('data/train/generated_qa_pairs.json')
# print('Deleted old QA pairs')


FileNotFoundError: [Errno 2] No such file or directory: 'data/train/generated_qa_pairs.json'

## Step 1 — Upload Your Code

Run the cell below. A file-picker will appear — select the **`homework_code.zip`** you created locally:

```bash
# Run this on your local machine first to create the zip:
cd /path/to/homework4_aug_4
zip -r homework_code.zip homework/ requirements.txt --exclude "*.pyc" "*/__pycache__/*"
```

In [4]:
from google.colab import files
import zipfile, os, sys, site
from pathlib import Path

uploaded = files.upload()   # pick homework_code.zip
zip_name = list(uploaded.keys())[0]

PROJECT = '/content/homework4'
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(PROJECT)

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Register path permanently so it survives kernel restarts in this session
pth = Path(site.getsitepackages()[0]) / 'homework4.pth'
pth.write_text(PROJECT + '\n')
print(f'Path registered: {pth}')

print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

Saving homework_code.zip to homework_code.zip
Path registered: /usr/local/lib/python3.12/dist-packages/homework4.pth
Working directory: /content/homework4
Contents: ['homework', 'requirements.txt', 'data', 'grader', 'bundle.py']


In [ ]:
# from google.colab import files
# import zipfile, os

# uploaded = files.upload()   # pick the new homework_code.zip
# zip_name = list(uploaded.keys())[0]

# # Extract only the data/ folder — leaves homework/vlm_model untouched
# with zipfile.ZipFile(zip_name, 'r') as z:
#     data_members = [m for m in z.namelist() if m.startswith('data/')]
#     z.extractall('/content/homework4', members=data_members)

# print('Extracted:', os.listdir('/content/homework4/data/'))


## Step 2 — Install Dependencies

Colab already has `torch` and `torchvision`. We only install the extras.

> If any cell below raises an `ImportError`, go to **Runtime → Restart session**, then run **Step 3** next (you can skip Steps 1–2 after a restart — files persist).

In [5]:
%pip install -q transformers==4.52.4 peft==0.16.0 fire tqdm opencv-python tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.2 MB/s eta 0:00:00


## Step 3 — Path Setup (re-run this after any runtime restart)

In [6]:
# import os, sys

# PROJECT = '/content/homework4'
# os.chdir(PROJECT)
# if PROJECT not in sys.path:
#     sys.path.insert(0, PROJECT)

# print('Working directory:', os.getcwd())

## Step 4 — Download Dataset

In [7]:
from pathlib import Path
import os

data_dir = Path('data/train')
if not data_dir.exists() or not any(data_dir.glob('*_info.json')):
    print('Downloading dataset...')
    os.system('wget -q https://utexas.box.com/shared/static/qubjm5isldqvyimfj9rsmbnvnbezwcv4.zip -O supertux_data.zip')
    os.system('unzip -q supertux_data.zip')
    os.remove('supertux_data.zip')
    print('Done.')
else:
    n = len(list(data_dir.glob('*_info.json')))
    print(f'Dataset already present ({n} info files in data/train)')

Done.


In [12]:
from google.colab import files
import zipfile

uploaded = files.upload()   # pick the new homework_code.zip
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    # Only extract homework/ and requirements — leave data/ and checkpoints alone
    members = [m for m in z.namelist()
               if m.startswith('homework/') or m.startswith('data/valid_grader')
               or m.startswith('data/train_demo') or m == 'requirements.txt']
    z.extractall('/content/homework4', members=members)

print('Updated files:')
for m in members[:10]:
    print(' ', m)
print(f'  ... ({len(members)} files total)')


Saving homework_code.zip to homework_code.zip
Updated files:
  homework/
  homework/generate_qa.py
  homework/base_vlm.py
  homework/demo_train/
  homework/demo_train/adapter_model.safetensors
  homework/demo_train/events.out.tfevents.1777027298.josephines-MacBook-Pro.local.8865.1
  homework/demo_train/events.out.tfevents.1777023871.josephines-MacBook-Pro.local.78735.1
  homework/demo_train/README.md
  homework/demo_train/checkpoint-5/
  homework/demo_train/checkpoint-5/adapter_model.safetensors
  ... (64 files total)


In [9]:
# import os
# os.remove('data/train/generated_qa_pairs.json')


---
## Part 1 — VLM Fine-Tuning

Target: **≥ 70% accuracy** on the validation set (50 pts).

### 1a — Generate QA Training Data

In [2]:
import json, random
from pathlib import Path
from homework.generate_qa import generate_qa_pairs

data_train = Path('data/train')
out_file = data_train / 'generated_qa_pairs.json'

if out_file.exists():
    print(f'QA file already exists: {out_file}')
else:
    # Build image name set once — avoids O(n²) directory scans in the loop
    existing_images = {f.name for f in data_train.glob('*_im.jpg')}
    print(f'Found {len(existing_images)} images, generating QA pairs...')

    all_qa = []
    for info_file in sorted(data_train.glob('*_info.json')):
        base = info_file.stem.replace('_info', '')
        with open(info_file) as f:
            info = json.load(f)
        num_views = len(info.get('detections', []))
        for view_index in range(num_views):
            img_name = f'{base}_{view_index:02d}_im.jpg'
            if img_name not in existing_images:
                continue
            try:
                pairs = generate_qa_pairs(str(info_file), view_index)
                for p in pairs:
                    p['image_file'] = f'train/{img_name}'
                all_qa.extend(pairs)
            except Exception:
                pass

    random.shuffle(all_qa)
    out_file.write_text(json.dumps(all_qa, indent=2))
    print(f'Generated {len(all_qa)} QA pairs → {out_file}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Found 52500 images, generating QA pairs...
Generated 452523 QA pairs → data/train/generated_qa_pairs.json


In [4]:
import gc, os
import torch
# ── Clear any leftover GPU state from previous runs ────────────────────────
for _var in ['model', 'trainer', 'vlm', 'train_dataset']:
    if _var in dir():
        del _var
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / '
      f'{torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

GPU free: 42.0 GB / 42.4 GB total


In [6]:
import shutil
from pathlib import Path
for ckpt in Path('homework/vlm_model').rglob('checkpoint-*'):
    shutil.rmtree(ckpt)
    print(f'Deleted {ckpt}')


Deleted homework/vlm_model/checkpoint-700
Deleted homework/vlm_model/checkpoint-650


### 1b — Train VLM

In [7]:
import gc, os
import torch
from pathlib import Path
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainerCallback, TrainingArguments

from homework.base_vlm import DEVICE, BaseVLM
from homework.data import VQADataset, benchmark
from homework.finetune import VQADatasetForTraining, custom_data_collator, processor

# ── Clear any leftover GPU state from previous runs ────────────────────────
for _var in ['model', 'trainer', 'vlm', 'train_dataset']:
    if _var in dir():
        del _var
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / '
      f'{torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

# ── Early-stopping callback ────────────────────────────────────────────────
class EarlyStopOnAccuracy(TrainerCallback):
    """Evaluates on valid_grader every eval_steps and stops when target_acc is met."""

    def __init__(self, vlm, target_acc=0.75, eval_steps=100, max_eval_samples=128):
        self.vlm              = vlm
        self.target_acc       = target_acc
        self.eval_steps       = eval_steps
        self.max_eval_samples = max_eval_samples
        self._eval_dataset    = None

    @property
    def eval_dataset(self):
        if self._eval_dataset is None:
            self._eval_dataset = VQADataset('valid_grader')
        return self._eval_dataset

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        self.vlm.model = model
        model.eval()
        result = benchmark(self.vlm, self.eval_dataset, self.max_eval_samples)
        print(f'\nStep {state.global_step} | val accuracy: {result.accuracy:.1%} | target: {self.target_acc:.0%}')
        model.train()

        if result.accuracy >= self.target_acc:
            print(f'Target {self.target_acc:.0%} reached at step {state.global_step} — stopping early.')
            control.should_training_stop = True

        return control

# ── Setup ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('homework/vlm_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vlm   = BaseVLM()
model = get_peft_model(
    vlm.model,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM, inference_mode=False,
        r=8, lora_alpha=32, lora_dropout=0.0,
        target_modules='all-linear', bias='none',
    ),
)
model.print_trainable_parameters()
model.config.use_cache = False
model.enable_input_require_grads()
model.train()

train_dataset = VQADatasetForTraining(VQADataset('train'), processor)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    logging_dir                 = str(OUTPUT_DIR),
    report_to                   = 'tensorboard',
    num_train_epochs            = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,
    gradient_checkpointing      = True,
    learning_rate               = 5e-4,        # halved — 5e-4 caused mode collapse
    warmup_ratio                = 0.05,        # gentle ramp-up to avoid early collapse
    weight_decay                = 0.01,        # regularisation
    lr_scheduler_type           = 'cosine',
    bf16                        = DEVICE == 'cuda',
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    label_names                 = ['labels'],
    dataloader_num_workers      = 2,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    data_collator = custom_data_collator,
    callbacks     = [EarlyStopOnAccuracy(vlm, target_acc=0.75, eval_steps=100, max_eval_samples=128)],
)

# ── Start fresh (collapsed checkpoints are not worth resuming) ────────────
print('Starting fresh — previous run collapsed, not resuming from checkpoint')
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
print(f'\nVLM saved to {OUTPUT_DIR}')

GPU free: 40.4 GB / 42.4 GB total
trainable params: 3,872,256 || all params: 260,357,184 || trainable%: 1.4873
Loaded 452523 QA pairs for train split
Starting fresh — previous run collapsed, not resuming from checkpoint


Step,Training Loss
1,9.283700
2,8.925600
3,9.431500
4,9.018600
5,8.406500
6,9.213100
7,8.085200
8,8.809600
9,8.873200
10,8.566700


Loaded 120 QA pairs for valid_grader split


 25%|██▌       | 1/4 [00:04<00:14,  4.88s/it]

	Processed 32 samples
	Questions: ['Where is kiki relative to the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'Is sara_the_wizard to the left or right of the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'What kart is the ego car?', 'How many karts are to the left of the ego car?', 'What kart is the ego car?', 'Where is hexley relative to the ego car?', 'How many karts are to the right of the ego car?', 'What track is this?', 'What kart is the ego car?', 'Where is emule relative to the ego car?', 'What track is this?', 'What kart is the ego car?', 'How many karts are behind the ego car?', 'How many karts are behind the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'What track is this?', 'Is puffy in front of or behind the ego car?', 'How many karts are in front of the ego car?', 'What track is this?', 'How ma

 50%|█████     | 2/4 [00:09<00:09,  4.79s/it]

... (42 outputs left out)

	Processed 96 samples
	Questions: ['How many karts are there in the scenario?', 'What track is this?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'How many karts are to the left of the ego car?', 'Is konqi to the left or right of the ego car?', 'Is beastie to the left or right of the ego car?', 'How many karts are there in the scenario?', 'How many karts are behind the ego car?', 'Is konqi in front of or behind the ego car?', 'How many karts are behind the ego car?', 'How many karts are there in the scenario?', 'How many karts are behind the ego car?', 'How many karts are in front of the ego car?', 'Where is puffy relative to the ego car?', 'What kart is the ego car?', 'What track is this?', 'What track is this?', 'What track is this?', 'What kart is the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'What track is this?', 'How many karts are behind the ego car?', 'What kart is the ego car?', 'What kart is the ego car

100%|██████████| 4/4 [00:17<00:00,  4.37s/it]

	Processed 120 samples
	Questions: ['How many karts are to the left of the ego car?', 'How many karts are behind the ego car?', 'Where is gavroche relative to the ego car?', 'What kart is the ego car?', 'What track is this?', 'What track is this?', 'How many karts are there in the scenario?', 'What track is this?', 'Is puffy in front of or behind the ego car?', 'What track is this?', 'What kart is the ego car?', 'Where is xue relative to the ego car?', 'How many karts are in front of the ego car?', 'Where is hexley relative to the ego car?', 'How many karts are there in the scenario?', 'How many karts are in front of the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'How many karts are behind the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'Where is konqi relative to the ego car?', 'How many karts are in front of the ego car?', 'How many karts are there in the scenario?']
	Responses: ['1', '0', 'front and right', 'gavroche', 'hacienda', '

KeyboardInterrupt: 

In [ ]:
# import os, sys
# os.chdir('/content/homework4')
# if '/content/homework4' not in sys.path:
#     sys.path.insert(0, '/content/homework4')



In [24]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [25]:
import shutil
shutil.copytree('homework/vlm_model', '/content/drive/MyDrive/hw4_vlm_model', dirs_exist_ok=True)
print('VLM checkpoint saved to Drive')


VLM checkpoint saved to Drive


### 1c — Evaluate VLM

In [26]:
from homework.finetune import load as vlm_load
from homework.data import VQADataset, benchmark

vlm     = vlm_load('vlm_model')
testset = VQADataset('valid_grader')
result  = benchmark(vlm, testset, 128)

print()
print(f'  VLM Accuracy : {result.accuracy:.1%}')
print(f'  Correct      : {int(result.accuracy * len(result.samples))} / {len(result.samples)}')
print(f'  Target       : 70.0%')
print(f'  Status       : {"✓  PASS — ready to submit" if result.accuracy >= 0.70 else "✗  below target — keep training"}')

Loaded 120 QA pairs for valid_grader split


 25%|██▌       | 1/4 [00:04<00:13,  4.64s/it]

	Processed 32 samples
	Questions: ['What kart is the ego car?', 'Is sara_the_racer in front of or behind the ego car?', 'What track is this?', 'How many karts are there in the scenario?', 'What track is this?', 'What track is this?', 'What track is this?', 'How many karts are there in the scenario?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'Where is hexley relative to the ego car?', 'How many karts are there in the scenario?', 'How many karts are to the right of the ego car?', 'Is amanda in front of or behind the ego car?', 'Where is xue relative to the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'How many karts are to the right of the ego car?', 'How many karts are in front of the ego car?', 'Is konqi in front of or behind the ego car?', 'How many karts are in front of the ego car?', 'Where is hexley relative to the ego car?', 'How many karts are behind 

 50%|█████     | 2/4 [00:09<00:09,  4.61s/it]

	Processed 64 samples
	Questions: ['How many karts are in front of the ego car?', 'Is sara_the_wizard to the left or right of the ego car?', 'How many karts are to the left of the ego car?', 'What track is this?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'What track is this?', 'Where is puffy relative to the ego car?', 'How many karts are there in the scenario?', 'Where is emule relative to the ego car?', 'Is gnu to the left or right of the ego car?', 'How many karts are behind the ego car?', 'How many karts are in front of the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'What track is this?', 'What kart is the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'Where is xue relative to the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'How many karts a

 75%|███████▌  | 3/4 [00:13<00:04,  4.62s/it]

	Processed 96 samples
	Questions: ['How many karts are there in the scenario?', 'How many karts are behind the ego car?', 'How many karts are there in the scenario?', 'Where is konqi relative to the ego car?', 'How many karts are to the left of the ego car?', 'How many karts are there in the scenario?', 'How many karts are there in the scenario?', 'What kart is the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'What kart is the ego car?', 'How many karts are to the left of the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'What track is this?', 'How many karts are in front of the ego car?', 'How many karts are in front of the ego car?', 'Where is puffy relative to the ego car?', 'Is suzanne to the left or right of the ego car?', 'How many karts are there in the scenario?', 'What track is this?', 'What kart is the ego car?', 'Is amanda in front of or behind the ego car?', 'What track is this?', 'How many karts

100%|██████████| 4/4 [00:17<00:00,  4.39s/it]

	Processed 120 samples
	Questions: ['What track is this?', 'Is hexley in front of or behind the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'How many karts are there in the scenario?', 'What kart is the ego car?', 'What kart is the ego car?', 'What kart is the ego car?', 'Where is kiki relative to the ego car?', 'How many karts are behind the ego car?', 'Is konqi to the left or right of the ego car?', 'What track is this?', 'What kart is the ego car?', 'Where is gavroche relative to the ego car?', 'How many karts are there in the scenario?', 'What track is this?', 'What kart is the ego car?', 'What track is this?', 'What track is this?', 'What track is this?', 'What track is this?', 'What kart is the ego car?', 'How many karts are behind the ego car?', 'Is konqi to the left or right of the ego car?']
	Responses: ['hacienda', 'front', 'konqi', 'wilber', '1', 'gnu', 'gavroche', 'konqi', 'front and left', '0', 'right', 'tutorial', 'sara_the_wizard', 'back and left

---
## Part 2 — CLIP

Target: **≥ 70% accuracy** on multi-choice QA (50 pts, random baseline is 20%).

### 2a — Generate Caption Training Data

In [27]:
import json, random
from pathlib import Path
from homework.generate_captions import generate_caption

data_train = Path('data/train')
out_file = data_train / 'generated_captions.json'

if out_file.exists():
    print(f'Captions file already exists: {out_file}')
else:
    # Build image name set once — avoids O(n²) directory scans in the loop
    existing_images = {f.name for f in data_train.glob('*_im.jpg')}
    print(f'Found {len(existing_images)} images, generating captions...')

    all_caps = []
    for info_file in sorted(data_train.glob('*_info.json')):
        base = info_file.stem.replace('_info', '')
        with open(info_file) as f:
            info = json.load(f)
        num_views = len(info.get('detections', []))
        for view_index in range(num_views):
            img_name = f'{base}_{view_index:02d}_im.jpg'
            if img_name not in existing_images:
                continue
            try:
                for cap in generate_caption(str(info_file), view_index):
                    all_caps.append({
                        'image_file': f'train/{img_name}',
                        'caption': cap,
                    })
            except Exception:
                pass

    random.shuffle(all_caps)
    out_file.write_text(json.dumps(all_caps, indent=2))
    print(f'Generated {len(all_caps)} captions → {out_file}')

Found 52500 images, generating captions...
Generated 185961 captions → data/train/generated_captions.json


### 2b — Train CLIP

In [28]:
import gc, os
import torch
import torchvision as tv
from pathlib import Path
from PIL import Image
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainerCallback, TrainingArguments

from homework.base_vlm import DEVICE as device, BaseVLM
from homework.clip import (
    CLIP, CaptionDatasetForTraining, clip_data_collator,
    compute_clip_loss, get_target_modules_for_lora, processor,
)
from homework.data import CaptionDataset, MultiChoiceQADataset

# # ── Clear GPU state ────────────────────────────────────────────────────────
# for _var in ['model', 'trainer', 'vlm']:
#     if _var in dir():
#         del _var
# gc.collect()
# torch.cuda.empty_cache()
# os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
# print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / '
#       f'{torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

# ── Early-stopping callback ────────────────────────────────────────────────
class ClipEarlyStop(TrainerCallback):
    """Evaluates CLIP on valid_grader every eval_steps; stops when target_acc is met."""

    def __init__(self, target_acc=0.75, eval_steps=50, max_eval_samples=256):
        self.target_acc       = target_acc
        self.eval_steps       = eval_steps
        self.max_eval_samples = max_eval_samples
        self._testset         = None
        self._image_proc      = tv.transforms.Compose([
            tv.transforms.Resize(192),
            tv.transforms.CenterCrop(192),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    @property
    def testset(self):
        if self._testset is None:
            self._testset = MultiChoiceQADataset('valid_grader')
        return self._testset

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        clip_model = model.model  # unwrap PeftModel → CLIP
        clip_model.eval()
        correct = total = 0

        with torch.no_grad():
            for i, pair in enumerate(self.testset):
                if i >= self.max_eval_samples:
                    break
                image = Image.open(pair['image_path']).convert('RGB')
                pv = self._image_proc(image).unsqueeze(0).to(device)
                if device == 'cuda':
                    pv = pv.bfloat16()
                ti = processor(
                    text=[s + processor.tokenizer.eos_token for s in pair['candidates']],
                    return_tensors='pt', padding=True, truncation=True,
                )
                ids  = ti['input_ids'].long().to(device)
                mask = ti['attention_mask'].to(device)
                vf, tf, _ = clip_model(pv, ids, mask)
                if torch.matmul(vf, tf.T).argmax(dim=-1).item() == pair['correct_index']:
                    correct += 1
                total += 1

        acc = correct / total if total else 0.0
        print(f'\nStep {state.global_step} | CLIP val accuracy: {acc:.1%} | target: {self.target_acc:.0%}')
        clip_model.train()

        if acc >= self.target_acc:
            print(f'Target {self.target_acc:.0%} reached at step {state.global_step} — stopping early.')
            control.should_training_stop = True

        return control

# ── Setup ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('homework/clip_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vlm            = BaseVLM()
vision_encoder = vlm.model.model.vision_model
text_encoder   = vlm.model.model.text_model
model          = CLIP(vision_encoder, text_encoder).to(device)
if device == 'cuda':
    model = model.bfloat16()
model.set_trainable_parameters()

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
    r=8, lora_alpha=32, lora_dropout=0.0,
    target_modules=get_target_modules_for_lora(model),
    bias='none',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.to(device)
model.train()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

train_dataset = CaptionDatasetForTraining(CaptionDataset('train'), processor)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    logging_dir                 = str(OUTPUT_DIR),
    report_to                   = 'tensorboard',
    num_train_epochs            = 3,
    per_device_train_batch_size = 256,
    gradient_accumulation_steps = 1,
    gradient_checkpointing      = True,
    learning_rate               = 5e-4,
    bf16                        = device == 'cuda',
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    label_names                 = ['labels'],
    dataloader_num_workers      = 2,
)

trainer = Trainer(
    model             = model,
    args              = training_args,
    train_dataset     = train_dataset,
    data_collator     = clip_data_collator,
    compute_loss_func = compute_clip_loss,
    callbacks         = [ClipEarlyStop(target_acc=0.75, eval_steps=50, max_eval_samples=256)],
)

# ── Auto-resume from latest checkpoint if one exists ──────────────────────
_checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[1]))
_resume = str(_checkpoints[-1]) if _checkpoints else None
print(f'Resuming from: {_resume}' if _resume else 'Starting fresh — no checkpoint found')

trainer.train(resume_from_checkpoint=_resume)

# Save model + projection head weights
trainer.save_model(str(OUTPUT_DIR))
model.model.save_pretrained(str(OUTPUT_DIR))
print(f'\nCLIP saved to {OUTPUT_DIR}')

GPU free: 41.0 GB / 42.4 GB total
trainable params: 3,769,344 || all params: 224,877,121 || trainable%: 1.6762
Loaded 185961 captions for train split
Starting fresh — no checkpoint found


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,5.734700
2,5.635300
3,5.592700
4,5.555600
5,5.552100
6,5.547200
7,5.546500
8,5.546900
9,5.546100
10,5.544700


Loaded 200 QA pairs for valid_grader split

Step 50 | CLIP val accuracy: 33.0% | target: 70%

Step 100 | CLIP val accuracy: 52.0% | target: 70%

Step 150 | CLIP val accuracy: 52.5% | target: 70%

Step 200 | CLIP val accuracy: 71.5% | target: 70%
Target 70% reached at step 200 — stopping early.

CLIP saved to homework/clip_model


In [29]:
# import shutil
shutil.copytree('homework/clip_model', '/content/drive/MyDrive/hw4_clip_model', dirs_exist_ok=True)
print('CLIP checkpoint saved to Drive')

CLIP checkpoint saved to Drive


### 2c — Evaluate CLIP

In [30]:
from homework.clip import test as clip_test

clip_test('clip_model')

Loaded 200 QA pairs for valid_grader split


100%|██████████| 200/200 [00:26<00:00,  7.65it/s]

Accuracy: 0.715


---
## Submission

Replace `UT_ID` with your actual UT EID, then run the cell. It will:
1. Strip intermediate checkpoints to stay under the 50 MB limit
2. Build the submission zip with `bundle.py`
3. Run the local grader to verify
4. Download the zip to your machine

In [33]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import files

UT_ID = 'ja56388'   # ← your UT EID

# Remove intermediate checkpoints to stay under 50 MB
for ckpt in Path('homework').rglob('checkpoint-*'):
    shutil.rmtree(ckpt, ignore_errors=True)
    print(f'Removed {ckpt}')

# Upload bundle.py + grader/ if they weren't in the original zip
if not Path('bundle.py').exists():
    print('bundle.py not found — upload it now (pick bundle.py from the file picker)')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
    print('Uploaded:', list(uploaded.keys()))

if not Path('grader').exists():
    print('grader/ not found — upload grader.zip or re-zip with the updated zip command')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
        if name.endswith('.zip'):
            import zipfile
            with zipfile.ZipFile(name) as z:
                z.extractall('.')
    print('Uploaded:', list(uploaded.keys()))

# Build submission zip — capture output so errors are visible
print('\n── Running bundle.py ──')
result = subprocess.run(
    ['python3', 'bundle.py', 'homework', UT_ID],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('BUNDLE ERROR:\n', result.stderr)
    raise RuntimeError('bundle.py failed — fix the error above before downloading')

zip_path = Path(f'{UT_ID}.zip')
if not zip_path.exists():
    raise FileNotFoundError(f'{zip_path} was not created — check bundle.py output above')

print(f'Zip size: {zip_path.stat().st_size / 1e6:.1f} MB')

# Verify with the local grader
print('\n── Running grader ──')
result = subprocess.run(
    ['python3', '-m', 'grader', str(zip_path)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('GRADER ERROR:\n', result.stderr)

# Download the zip to your local machine
files.download(str(zip_path))

bundle.py not found — upload it now (pick bundle.py from the file picker)


Saving homework_code.zip to homework_code.zip
Uploaded: ['homework_code.zip']
grader/ not found — upload grader.zip or re-zip with the updated zip command


Saving homework_code.zip to homework_code (1).zip
Uploaded: ['homework_code (1).zip']

── Running bundle.py ──
clip_model
data.py
generate_captions.py
finetune.py
vlm_model
__init__.py
clip.py
generate_qa.py
base_vlm.py
demo_train
clip_model/training_args.bin
clip_model/README.md
clip_model/additional_weights.pt
clip_model/adapter_model.safetensors
clip_model/adapter_config.json
clip_model/events.out.tfevents.1777178164.a5a5d5e3d01a.2068.2
vlm_model/training_args.bin
vlm_model/tensorboard
vlm_model/events.out.tfevents.1777137454.a5a5d5e3d01a.2068.0
vlm_model/README.md
vlm_model/events.out.tfevents.1777157146.a5a5d5e3d01a.2068.1
vlm_model/events.out.tfevents.1777112579.josephines-MacBook-Pro.local.89163.1
vlm_model/checkpoint-200
vlm_model/adapter_model.safetensors
vlm_model/adapter_config.json
vlm_model/checkpoint-150
vlm_model/events.out.tfevents.1777112022.josephines-MacBook-Pro.local.87114.1
demo_train/events.out.tfevents.1777023871.josephines-MacBook-Pro.local.78735.1
demo_train/tr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import files

UT_ID = 'ja56388'   # ← your UT EID

homework = Path('homework')

# ── Remove checkpoint folders ─────────────────────────────────────────────
for ckpt in homework.rglob('checkpoint-*'):
    shutil.rmtree(ckpt, ignore_errors=True)
    print(f'Removed {ckpt}')

# ── Remove demo_train (not part of submission) ────────────────────────────
demo = homework / 'demo_train'
if demo.exists():
    shutil.rmtree(demo)
    print(f'Removed {demo}')

# ── Remove TensorBoard event files ────────────────────────────────────────
for f in homework.rglob('events.out.tfevents.*'):
    f.unlink()
    print(f'Removed {f}')
for tb_dir in homework.rglob('tensorboard'):
    shutil.rmtree(tb_dir, ignore_errors=True)
    print(f'Removed {tb_dir}')

# ── Remove demo_clip if it exists ─────────────────────────────────────────
demo_clip = homework / 'demo_clip'
if demo_clip.exists():
    shutil.rmtree(demo_clip)
    print(f'Removed {demo_clip}')

# ── Upload bundle.py + grader/ if they weren't in the original zip ────────
if not Path('bundle.py').exists():
    print('bundle.py not found — upload it now')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)

if not Path('grader').exists():
    print('grader/ not found — upload grader.zip')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
        if name.endswith('.zip'):
            import zipfile
            with zipfile.ZipFile(name) as z:
                z.extractall('.')

# ── Build submission zip ──────────────────────────────────────────────────
print('\n── Running bundle.py ──')
result = subprocess.run(
    ['python3', 'bundle.py', 'homework', UT_ID],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('BUNDLE ERROR:\n', result.stderr)
    raise RuntimeError('bundle.py failed — fix the error above before downloading')

zip_path = Path(f'{UT_ID}.zip')
if not zip_path.exists():
    raise FileNotFoundError(f'{zip_path} was not created — check bundle.py output above')

print(f'Zip size: {zip_path.stat().st_size / 1e6:.1f} MB')

# ── Verify with the local grader ──────────────────────────────────────────
print('\n── Running grader ──')
result = subprocess.run(
    ['python3', '-m', 'grader', str(zip_path)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('GRADER ERROR:\n', result.stderr)

# ── Download to PC ────────────────────────────────────────────────────────
files.download(str(zip_path))

Removed homework/vlm_model/checkpoint-200
Removed homework/vlm_model/checkpoint-150
Removed homework/demo_train/checkpoint-5
Removed homework/demo_train
Removed homework/clip_model/events.out.tfevents.1777178164.a5a5d5e3d01a.2068.2
Removed homework/vlm_model/events.out.tfevents.1777137454.a5a5d5e3d01a.2068.0
Removed homework/vlm_model/events.out.tfevents.1777157146.a5a5d5e3d01a.2068.1
Removed homework/vlm_model/events.out.tfevents.1777112579.josephines-MacBook-Pro.local.89163.1
Removed homework/vlm_model/events.out.tfevents.1777112022.josephines-MacBook-Pro.local.87114.1
Removed homework/vlm_model/tensorboard/events.out.tfevents.1777111285.josephines-MacBook-Pro.local.84600.0
Removed homework/vlm_model/tensorboard/events.out.tfevents.1777112020.josephines-MacBook-Pro.local.87114.0
Removed homework/vlm_model/tensorboard/events.out.tfevents.1777111866.josephines-MacBook-Pro.local.86548.0
Removed homework/vlm_model/tensorboard/events.out.tfevents.1777112577.josephines-MacBook-Pro.local.89

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>